# Pilot 2

- Upbringing (Good/Bad) × Belief (Activist/Bigot) × Action (Help/Harm)
- Responsibility for Action (–6 to +6), Blame/Praise (–6 to +6)
- N = 481

## Set Up Notebook

In [ ]:
##########
# IMPORT #
##########

# Data processing and math
import pandas as pd
import numpy as np

# Statistics
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display handling
import warnings


#################
# CONFIGURATION #
#################

# Suppress warnings
warnings.filterwarnings('ignore')

# Set global plotting style
sns.set_theme(style = "whitegrid")


##########
# LABELS #
##########
labels_measure = {
    'measure_action': 'Responsibility for Action',
    'measure_blame':  'Blame / Praise'
}


#################
# VISUALIZATION #
#################
palette_main = {
    "Good":      "#0072B2", "Bad":         "#D55E00",
    "Activist":  "#56B4E9", "Bigot":       "#E69F00",
    "Help":      "#F0E442", "Harm":        "#882255"
}

## Transform Data

In [ ]:
# Load data
df = pd.read_csv('blame_praise_self_pilot_2_data.csv')

# Define factors
factors_map = {
     5: {'Upbringing': 'Bad',  'Belief': 'Activist', 'Action': 'Help'},
     6: {'Upbringing': 'Bad',  'Belief': 'Activist', 'Action': 'Harm'},
     7: {'Upbringing': 'Bad',  'Belief': 'Bigot',    'Action': 'Help'},
     8: {'Upbringing': 'Bad',  'Belief': 'Bigot',    'Action': 'Harm'},
    13: {'Upbringing': 'Good', 'Belief': 'Activist', 'Action': 'Help'},
    14: {'Upbringing': 'Good', 'Belief': 'Activist', 'Action': 'Harm'},
    15: {'Upbringing': 'Good', 'Belief': 'Bigot',    'Action': 'Help'},
    16: {'Upbringing': 'Good', 'Belief': 'Bigot',    'Action': 'Harm'}
}

# Reshape wide to long
indices = [5, 6, 7, 8, 13, 14, 15, 16]
long_data = []

for _, row in df.iterrows():
    p_id = row['ID']
    
    for i in indices:
        col_action = f'homophobia.{i}-pAction'
        
        col_blame = None
        search_pattern = f'blame{i}' if i != 13 else 'blame.13'
        for col in df.columns:
            if search_pattern in col.replace(' ', ''):
                col_blame = col
                break
        
        if col_action in df.columns and pd.notna(row[col_action]):
            factors = factors_map[i]
            
            score_action = pd.to_numeric(row[col_action], errors = 'coerce') - 7
            
            score_blame = np.nan
            if col_blame and pd.notna(row[col_blame]):
                score_blame = pd.to_numeric(row[col_blame], errors = 'coerce') - 7
            
            long_data.append({
                'ID': p_id,
                'Upbringing': factors['Upbringing'],
                'Belief': factors['Belief'],
                'Action': factors['Action'],
                'measure_action': score_action,
                'measure_blame': score_blame,
                'Condition_Index': i
            })

df_long = pd.DataFrame(long_data)
print(f"Transformation complete ({len(df_long)} Observations).")

## Calculate Descriptive Statistics

In [ ]:
# Define group factors and dependent variables
group_factors = ['Upbringing', 'Belief', 'Action']
dependent_vars = ['measure_action', 'measure_blame']

# Calculate descriptive statistics
descriptive_stats = df_long.groupby(group_factors)[dependent_vars].agg(['mean', 'std', 'count']).round(3)
total_participants = descriptive_stats[('measure_action', 'count')].sum()

# Display results
display(descriptive_stats)
display(f"N = {total_participants}")

## Perform ANOVAs

In [ ]:
for dv in ['measure_action', 'measure_blame']:
    print(f"\nANOVA ({labels_measure.get(dv, dv)})")
    
    # Define formula
    formula = f"{dv} ~ C(Upbringing) * C(Belief) * C(Action)"
    
    # Drop lines with NaN
    df_anova = df_long.dropna(subset=[dv])

    # Fit model
    model = ols(formula, data = df_anova).fit()
    
    # Run ANOVA
    aov_table = sm.stats.anova_lm(model, typ = 2)
    
    # Calculate effect sizes
    aov_table['partial_eta_sq'] = aov_table['sum_sq'] / (aov_table['sum_sq'] + aov_table.loc['Residual', 'sum_sq'])
    
    # Display results
    display(aov_table.round(3))

## Generate Histograms

In [ ]:
df_long['Condition'] = (df_long['Upbringing'] + " + " + 
                        df_long['Belief'] + " + " + 
                        df_long['Action'])

for dv in dependent_vars:
    # Generate graph
    g = sns.displot(data = df_long,
                    x = dv,
                    col = "Condition",
                    col_wrap = 4,
                    kind = "hist",
                    discrete = True,
                    shrink = 0.8,
                    height = 3,
                    aspect = 1.2,
                    color = "#0072B2"
                   )
    
    # Set labels and titles
    g.set_axis_labels("Value", "Count")
    g.set_titles("{col_name}")
    
    # Limit x-axis
    for ax in g.axes.flat:
        ax.set_xticks(range(-6, 7))
        ax.set_xlim(-6.5, 6.5)
    
    # Add title
    plt.subplots_adjust(top = 0.8)
    current_label = labels_measure.get(dv, dv)
    g.fig.suptitle(f'Distribution: {current_label}', fontsize = 16)
    
    # Display graph
    plt.show()

## Generate Bar Plots

In [ ]:
for measure in ['measure_action', 'measure_blame']:
    current_label = labels_measure.get(measure, measure)
    
    # Create graphs
    g = sns.catplot(data = df_long,
                    x = "Upbringing",
                    y = measure,
                    hue = "Belief",
                    col = "Action",
                    kind = "bar",
                    palette = palette_main,
                    capsize = .05,
                    height = 5
                   )
    
    # Set axis labels and titles
    g.set_axis_labels("Upbringing", f"Mean (–6 to +6)")
    g.set_titles("{col_name}")
    
    # Add horizontal zero lines
    for ax in g.axes.flat:
        ax.axhline(0, color = 'black', lw = 1)
        ax.set_ylim(-6, 6)
    
    # Add main title
    plt.subplots_adjust(top = 0.85)
    g.fig.suptitle(current_label, fontsize = 16)
    
    # Export graphs
    filename = f'blame_praise_self_pilot_2_bar_plot_{measure}.png'
    g.savefig(filename, dpi = 300, bbox_inches = 'tight')
    plt.show()
    print(f"Graph saved as '{filename}'.")